# 01 — Data exploration

Reads `data/processed/{train,val,test}.jsonl` written by `make data` / `make synth`. All numbers here are recomputed from the files; the canonical tables live in `reports/data_stats.md`.

In [1]:
import json
import os
from collections import Counter
from pathlib import Path

import pandas as pd

from receipt_vlm.utils.io import PROJECT_ROOT, read_jsonl

os.chdir(PROJECT_ROOT)
rows = [r for s in ("train", "val", "test") for r in read_jsonl(f"data/processed/{s}.jsonl")]
df = pd.DataFrame(rows)
print(len(df), "samples")

621 samples


## Counts per source × split and language × split

In [2]:
display(pd.crosstab(df.source, df.split, margins=True))
display(pd.crosstab(df.language, df.split, margins=True))

split,test,train,val,All
source,,,,
sroie,62,497,62,621
All,62,497,62,621


split,test,train,val,All
language,,,,
en,62,497,62,621
All,62,497,62,621


## Field label availability (share of samples whose source labels the field)

In [3]:
from receipt_vlm.data.record import LABEL_KEYS

avail = pd.DataFrame({k: df.labelled.apply(lambda labs, k=k: k in labs) for k in LABEL_KEYS})
avail["source"] = df.source
display((avail.groupby("source").mean() * 100).round(0).T)

source,sroie
merchant,100.0
date,100.0
time,0.0
currency,0.0
subtotal,0.0
tax,0.0
total,100.0
payment_method,0.0
language,0.0
items.name,0.0


## Value distributions for labelled fields

In [4]:
tgt = pd.json_normalize(df.target)
tgt["source"] = df.source.values
print(tgt.groupby("source")["total"].describe())
print("dates by year:", Counter(d[:4] for d in tgt.date.dropna()).most_common())
print(
    "items per receipt:", df.target.apply(lambda t: len(t["items"])).groupby(df.source).describe()
)

        count       mean         std   min   25%   50%     75%     max
source                                                                
sroie   620.0  69.491097  327.577214 -1.73  8.85  27.0  60.375  7838.8
dates by year: [('2018', 395), ('2017', 188), ('2016', 34), ('2019', 4)]
items per receipt:         count  mean  std  min  25%  50%  75%  max
source                                           
sroie   621.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0


## A few images per source

In [5]:
import matplotlib.pyplot as plt
from PIL import Image

srcs = sorted(df.source.unique())
fig, axes = plt.subplots(len(srcs), 4, figsize=(12, 4.5 * len(srcs)), squeeze=False)
for i, s in enumerate(srcs):
    sub = (
        df[df.source == s].sample(4, random_state=0)
        if (df.source == s).sum() >= 4
        else df[df.source == s]
    )
    for j, (_, r) in enumerate(sub.iterrows()):
        im = Image.open(r.image)
        im.thumbnail((400, 600))
        axes[i, j].imshow(im)
        axes[i, j].set_title(f"{r.id}\ntotal={r.target['total']}", fontsize=8)
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()

<figure omitted from the committed notebook; re-run to view>